# AI-Powered Customer Churn Intelligence System

## Day 3 - SQL Business Analysis

This notebook creates a SQLite analytical database from the processed customer churn dataset and performs SQL-based business analysis.

The analysis focuses on:

- Customer churn
- Subscription plans
- Customer engagement
- Support activity
- Payment failures
- Customer tenure
- Usage behavior
- Churn-risk segmentation

In [18]:
import pandas as pd
import sqlite3
from pathlib import Path

In [19]:
# Load processed dataset

data_path = "../data/customer_churn_processed.csv"

df = pd.read_csv(data_path)

print("Processed dataset loaded successfully.")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Processed dataset loaded successfully.
Rows: 2,800
Columns: 15


## 1. Create SQLite Analytical Database

The processed customer dataset is loaded into a SQLite database to support reusable SQL-based business analysis.

In [20]:
# Create SQLite database

database_path = "../data/customer_churn.db"

conn = sqlite3.connect(database_path)

print(f"SQLite database created: {database_path}")

SQLite database created: ../data/customer_churn.db


In [21]:
# Create customer_churn table

df.to_sql(
    "customer_churn",
    conn,
    if_exists="replace",
    index=False
)

print("Table 'customer_churn' created successfully.")

Table 'customer_churn' created successfully.


In [22]:
# Verify table structure

table_info = pd.read_sql_query(
    "PRAGMA table_info(customer_churn);",
    conn
)

table_info

,cid,name,type,notnull,dflt_value,pk
0,0,user_id,INTEGER,0,None,0
1,1,signup_date,TEXT,0,None,0
2,2,plan_type,TEXT,0,None,0
3,3,monthly_fee,INTEGER,0,None,0
4,4,avg_weekly_usage_hours,REAL,0,None,0
5,5,support_tickets,INTEGER,0,None,0
6,6,payment_failures,INTEGER,0,None,0
7,7,tenure_months,INTEGER,0,None,0
8,8,last_login_days_ago,INTEGER,0,None,0
9,9,churn,TEXT,0,None,0


In [23]:
# Verify record count

record_count = pd.read_sql_query(
    "SELECT COUNT(*) AS total_records FROM customer_churn;",
    conn
)

record_count

,total_records
0,2800


## 2. Overall Churn Performance

The first analysis establishes the overall customer base, number of churned customers, retained customers, and overall churn rate.

In [24]:
query = """
SELECT
    COUNT(*) AS total_customers,
    SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    SUM(CASE WHEN churn = 'No' THEN 1 ELSE 0 END) AS retained_customers,
    ROUND(
        100.0 * SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS churn_rate_percentage
FROM customer_churn;
"""

overall_churn = pd.read_sql_query(query, conn)

overall_churn

,total_customers,churned_customers,retained_customers,churn_rate_percentage
0,2800,1605,1195,57.32


## 3. Churn by Subscription Plan

This analysis compares customer churn across subscription plans to identify which plan has the highest churn rate.

In [25]:
query_plan_churn = """
SELECT
    plan_type,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    SUM(CASE WHEN churn = 'No' THEN 1 ELSE 0 END) AS retained_customers,
    ROUND(
        100.0 * SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS churn_rate_percentage
FROM customer_churn
GROUP BY plan_type
ORDER BY churn_rate_percentage DESC;
"""

plan_churn = pd.read_sql_query(query_plan_churn, conn)

plan_churn

,plan_type,total_customers,churned_customers,retained_customers,churn_rate_percentage
0,Premium,944,548,396,58.05
1,Basic,923,534,389,57.85
2,Standard,933,523,410,56.06


## 4. Churn by Login Engagement

This analysis examines whether customers who have been inactive for longer periods show higher churn rates.

In [26]:
query_login_churn = """
SELECT
    login_recency_category,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS churn_rate_percentage
FROM customer_churn
GROUP BY login_recency_category
ORDER BY churn_rate_percentage DESC;
"""

login_churn = pd.read_sql_query(query_login_churn, conn)

login_churn

,login_recency_category,total_customers,churned_customers,churn_rate_percentage
0,Inactive,1350,891,66.00
1,At Risk,741,417,56.28
2,Active,709,297,41.89


## 5. Churn by Support Risk

This analysis examines the relationship between support-ticket activity and customer churn.

In [27]:
query_support_churn = """
SELECT
    support_risk,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS churn_rate_percentage
FROM customer_churn
GROUP BY support_risk
ORDER BY churn_rate_percentage DESC;
"""

support_churn = pd.read_sql_query(query_support_churn, conn)

support_churn

,support_risk,total_customers,churned_customers,churn_rate_percentage
0,High,924,603,65.26
1,Medium,882,512,58.05
2,Low,994,490,49.30


## 6. Churn by Payment Risk

This analysis evaluates whether customers experiencing payment failures have higher churn rates.

In [28]:
query_payment_churn = """
SELECT
    payment_risk,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS churn_rate_percentage
FROM customer_churn
GROUP BY payment_risk
ORDER BY churn_rate_percentage DESC;
"""

payment_churn = pd.read_sql_query(query_payment_churn, conn)

payment_churn

,payment_risk,total_customers,churned_customers,churn_rate_percentage
0,High,1384,924,66.76
1,Medium,974,506,51.95
2,Low,442,175,39.59


## 7. Churn by Usage Level

This analysis examines whether customer usage intensity is associated with different churn rates.

In [29]:
query_usage_churn = """
SELECT
    usage_level,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS churn_rate_percentage
FROM customer_churn
GROUP BY usage_level
ORDER BY churn_rate_percentage DESC;
"""

usage_churn = pd.read_sql_query(query_usage_churn, conn)

usage_churn

,usage_level,total_customers,churned_customers,churn_rate_percentage
0,Low,523,399,76.29
1,High,1177,631,53.61
2,Medium,1100,575,52.27


## 8. Churn by Customer Tenure

This analysis examines the relationship between customer tenure and churn. Customers are grouped into tenure bands to make the results easier to interpret.

In [30]:
query_tenure_churn = """
SELECT
    CASE
        WHEN tenure_months <= 6 THEN '0-6 Months'
        WHEN tenure_months <= 12 THEN '7-12 Months'
        WHEN tenure_months <= 24 THEN '13-24 Months'
        ELSE '25+ Months'
    END AS tenure_group,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS churn_rate_percentage
FROM customer_churn
GROUP BY tenure_group
ORDER BY churn_rate_percentage DESC;
"""

tenure_churn = pd.read_sql_query(query_tenure_churn, conn)

tenure_churn

,tenure_group,total_customers,churned_customers,churn_rate_percentage
0,13-24 Months,921,535,58.09
1,25+ Months,943,545,57.79
2,0-6 Months,442,254,57.47
3,7-12 Months,494,271,54.86


## 9. High-Risk Customer Segment

Customers showing multiple behavioral risk indicators are grouped into a high-risk segment.

A customer is classified as high risk when all three conditions are satisfied:

- Inactive login status
- High payment risk
- High support risk

This rule-based segment will later provide a business benchmark for comparison with the machine learning churn model.

In [31]:
query_high_risk = """
SELECT
    CASE
        WHEN login_recency_category = 'Inactive'
             AND payment_risk = 'High'
             AND support_risk = 'High'
        THEN 'High Risk'
        ELSE 'Other Customers'
    END AS risk_segment,

    COUNT(*) AS total_customers,

    SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,

    ROUND(
        100.0 * SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS churn_rate_percentage

FROM customer_churn

GROUP BY risk_segment

ORDER BY churn_rate_percentage DESC;
"""

high_risk_segment = pd.read_sql_query(query_high_risk, conn)

high_risk_segment

,risk_segment,total_customers,churned_customers,churn_rate_percentage
0,High Risk,200,166,83.00
1,Other Customers,2600,1439,55.35


## 10. High-Risk Customer Identification

This query identifies individual customers who meet all three high-risk conditions and can therefore be prioritized for retention analysis.

In [32]:
query_high_risk_customers = """
SELECT
    user_id,
    plan_type,
    monthly_fee,
    avg_weekly_usage_hours,
    support_tickets,
    payment_failures,
    tenure_months,
    last_login_days_ago,
    churn
FROM customer_churn
WHERE login_recency_category = 'Inactive'
  AND payment_risk = 'High'
  AND support_risk = 'High'
ORDER BY
    payment_failures DESC,
    support_tickets DESC,
    last_login_days_ago DESC;
"""

high_risk_customers = pd.read_sql_query(
    query_high_risk_customers,
    conn
)

high_risk_customers.head(20)

,user_id,plan_type,monthly_fee,avg_weekly_usage_hours,support_tickets,payment_failures,tenure_months,last_login_days_ago,churn
0,1630,Basic,199,7.0,8,5,27,59,Yes
1,198,Premium,699,9.9,8,5,33,57,Yes
2,1376,Basic,199,24.3,8,5,17,55,No
3,1649,Premium,699,5.2,8,5,21,53,No
4,396,Standard,399,23.8,8,5,18,52,Yes
5,914,Premium,699,12.0,8,5,8,52,Yes
6,1306,Standard,399,13.2,8,5,5,52,No
7,1443,Standard,399,12.3,8,5,27,50,Yes
8,1619,Premium,699,20.3,8,5,14,49,Yes
9,2349,Premium,699,20.3,8,5,5,49,Yes


## 11. Customer Risk Indicator Summary

This analysis summarizes the main behavioral indicators associated with customer churn and provides a consolidated view for business decision-making.

In [33]:
risk_summary_query = """
SELECT
    COUNT(*) AS total_customers,

    ROUND(
        100.0 * SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS overall_churn_rate,

    ROUND(AVG(support_tickets), 2) AS avg_support_tickets,

    ROUND(AVG(payment_failures), 2) AS avg_payment_failures,

    ROUND(AVG(avg_weekly_usage_hours), 2) AS avg_weekly_usage_hours,

    ROUND(AVG(last_login_days_ago), 2) AS avg_days_since_login

FROM customer_churn;
"""

risk_summary = pd.read_sql_query(
    risk_summary_query,
    conn
)

risk_summary

,total_customers,overall_churn_rate,avg_support_tickets,avg_payment_failures,avg_weekly_usage_hours,avg_days_since_login
0,2800,57.32,3.89,2.49,12.89,30.01


## 12. Database Connection

The SQLite connection is closed after completing the SQL analysis.

In [34]:
conn.close()

print("SQLite database connection closed successfully.")

SQLite database connection closed successfully.
